# Experimentos del proyecto ECU 911

En este notebook registro los cuatro modelos que probé en la Semana 10. Uso la muestra de seis meses (abril a septiembre de 2026) y la división temporal del pipeline, porque el objetivo es predecir emergencias futuras.

In [1]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.entities import ViewType
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from pipeline_proyecto import RUTA_DATOS, cargar_y_dividir, construir_preparacion

logging.getLogger("mlflow").setLevel(logging.ERROR)
EXPERIMENTO = "ecu911-emergencias-6-meses"

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()
fechas = pd.read_csv(RUTA_DATOS, usecols=["fecha"], parse_dates=["fecha"])["fecha"].drop_duplicates().sort_values().reset_index(drop=True)
fecha_corte = fechas.iloc[int(len(fechas) * 0.8)]

with open(RUTA_DATOS, "rb") as archivo:
    huella_datos = hashlib.md5(archivo.read()).hexdigest()[:12]

print("Archivo:", RUTA_DATOS)
print("Periodo:", fechas.min().date(), "a", fechas.max().date())
print("Fecha de corte:", fecha_corte.date())
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)
print("Huella:", huella_datos)

Archivo: /workspaces/Data_science_end_to_end/laboratorios/data/obt_emergencias_muestra_6_meses.csv
Periodo: 2026-04-01 a 2026-09-30
Fecha de corte: 2026-08-25
Entrenamiento: (32256, 11) Prueba: (8064, 11)
Huella: c0c505ad3ef8


## Registro limpio

Como este experimento es solo del proyecto, antes de la corrida final elimino sus pruebas anteriores. Así el CSV queda con una corrida completa por modelo y se puede volver a generar sin duplicados.

In [2]:
experimento_anterior = mlflow.get_experiment_by_name(EXPERIMENTO)
cliente = mlflow.MlflowClient()
if experimento_anterior is not None:
    anteriores = cliente.search_runs(
        [experimento_anterior.experiment_id],
        run_view_type=ViewType.ACTIVE_ONLY,
    )
    for corrida in anteriores:
        cliente.delete_run(corrida.info.run_id)

mlflow.set_experiment(EXPERIMENTO)
print("Registro de MLflow:", mlflow.get_tracking_uri())

Registro de MLflow: http://127.0.0.1:5050


In [3]:
def evaluar(nombre, modelo):
    pipeline = Pipeline([
        ("preparar", construir_preparacion()),
        ("modelo", modelo),
    ])

    with mlflow.start_run(run_name=nombre):
        parametros = modelo.get_params()
        parametros.update({
            "tipo_division": "temporal",
            "fecha_corte": str(fecha_corte.date()),
            "huella_datos": huella_datos,
            "filas_entrenamiento": len(X_ent),
            "filas_prueba": len(X_prueba),
        })
        mlflow.log_params(parametros)

        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
        mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))

        mlflow.log_metrics({
            "mae_entrenamiento": mae_ent,
            "mae_prueba": mae_prueba,
            "segundos": segundos,
        })
        mlflow.sklearn.log_model(pipeline, "modelo")

    print(f"{nombre:20s} MAE ent. {mae_ent:7.4f}  MAE prueba {mae_prueba:7.4f}  tiempo {segundos:6.2f}s")

## Los cuatro modelos

Mantengo los mismos hiperparámetros usados en los ejercicios anteriores para que la comparación sea justa.

In [4]:
evaluar("linea base", DummyRegressor(strategy="median"))
evaluar("ridge", Ridge(alpha=10.0, solver="lsqr"))
evaluar("bosque aleatorio", RandomForestRegressor(
    n_estimators=100, max_depth=14, min_samples_leaf=2,
    random_state=42, n_jobs=1,
))
evaluar("red neuronal", MLPRegressor(
    hidden_layer_sizes=(64,), max_iter=300,
    early_stopping=True, random_state=42,
))

linea base           MAE ent. 36.6786  MAE prueba 37.2822  tiempo   0.06s


ridge                MAE ent.  6.3885  MAE prueba  5.9268  tiempo   0.07s


bosque aleatorio     MAE ent.  3.5108  MAE prueba  5.1231  tiempo  34.67s


red neuronal         MAE ent.  5.4637  MAE prueba  4.9613  tiempo  24.01s


## Elección y reproducción

Ordeno las corridas por MAE de prueba. Marco como final la primera y cargo el artefacto guardado, no vuelvo a entrenarlo.

In [5]:
experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
corridas = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    order_by=["metrics.mae_prueba ASC"],
).reset_index(drop=True)

mejor_run_id = corridas.loc[0, "run_id"]
cliente.set_tag(mejor_run_id, "candidato", "final")
modelo_final = mlflow.sklearn.load_model(f"runs:/{mejor_run_id}/modelo")
mae_reproducido = mean_absolute_error(y_prueba, modelo_final.predict(X_prueba))

# Vuelvo a consultar para que la etiqueta final también salga en el CSV.
corridas = mlflow.search_runs(
    experiment_ids=[experimento.experiment_id],
    order_by=["metrics.mae_prueba ASC"],
).reset_index(drop=True)
corridas.to_csv("corridas_proyecto.csv", index=False)

columnas = ["run_id", "tags.mlflow.runName", "tags.candidato", "metrics.mae_entrenamiento", "metrics.mae_prueba", "metrics.segundos"]
print(corridas[columnas].to_string(index=False))
print()
print("MAE reproducido:", round(mae_reproducido, 10))
print("Coincide:", abs(mae_reproducido - corridas.loc[0, "metrics.mae_prueba"]) < 1e-9)

                          run_id tags.mlflow.runName tags.candidato  metrics.mae_entrenamiento  metrics.mae_prueba  metrics.segundos
cfe8821f136546399f2c313a53586424        red neuronal          final                   5.463667            4.961345         24.010051
e60512928d4f4d93afec8350fe9f1fc3    bosque aleatorio           None                   3.510838            5.123066         34.665663
312a85d8355a43678f4bb0a4685a855e               ridge           None                   6.388453            5.926758          0.071061
afc1114129b54d1b985b6661d640c138          linea base           None                  36.678571           37.282242          0.061356

MAE reproducido: 4.9613453737
Coincide: True


## Conclusión

El modelo final se escogió por el menor MAE de prueba en fechas posteriores al entrenamiento. La huella del CSV, la fecha de corte, los hiperparámetros y el pipeline quedaron registrados. Esto permite revisar de dónde salió el resultado y volver a cargar exactamente el artefacto entrenado.